<a href="https://colab.research.google.com/github/yt7360354-afk/vison_ai/blob/main/Week1_1_CNN_Image_Representation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎯 1주차 1-1. CNN 패러다임과 이미지 표현

> **시각지능(Vision AI) 고급 과정**
> **소요 시간:** 4시간 (이론 + 실습)
> **선수 지식:** Python, NumPy 기초, PyTorch 기본

---

## 📚 학습 목표

이 실습을 마치면 여러분은 다음을 할 수 있게 됩니다.

1. **디지털 이미지의 본질**을 픽셀·채널·텐서 관점에서 설명할 수 있다.
2. **NumPy/PyTorch 텐서**로 이미지를 자유롭게 변환·조작할 수 있다.
3. **이미지 전처리**(정규화·증강) 파이프라인을 직접 구축할 수 있다.
4. **MLP와 CNN의 본질적 차이**를 Spatial Locality 관점에서 설명하고, 실험으로 검증할 수 있다.

## 🗂️ 차시 구성

| 부 | 주제 | 소요시간 |
|---|---|---|
| 1부 | 디지털 이미지의 본질: 픽셀과 채널 | 60분 |
| 2부 | 텐서로서의 이미지 표현 | 60분 |
| 3부 | 이미지 전처리: 정규화와 증강 | 60분 |
| 4부 | MLP vs CNN: Spatial Locality | 60분 |

## 💡 실습 환경

- Google Colab (GPU 권장: 런타임 → 런타임 유형 변경 → T4 GPU)
- 필요한 패키지: `torch`, `torchvision`, `numpy`, `matplotlib`, `Pillow`, `albumentations`, `opencv-python`


---
## 0. 환경 설정

먼저 실습에 필요한 라이브러리를 설치하고 임포트합니다.

In [ ]:
# Colab 환경에서 추가로 필요한 라이브러리 설치
!pip install -q albumentations==1.4.18

# 실습 결과 재현성을 위한 시드 고정
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ 실습 디바이스: {device}")
print(f"✅ PyTorch 버전: {torch.__version__}")
print(f"✅ NumPy 버전: {np.__version__}")


In [ ]:
# 핵심 라이브러리 임포트
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import urllib.request
import io

# Matplotlib 한글 깨짐 방지 (Colab)
import matplotlib as mpl
mpl.rcParams['axes.unicode_minus'] = False

# Colab에서 한글 폰트 설치 (한 번만 실행)
import subprocess
try:
    subprocess.run(['apt-get', '-qq', 'install', 'fonts-nanum'], check=False)
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
    print("✅ 한글 폰트 설정 완료")
except Exception as e:
    print(f"한글 폰트 설정 생략: {e}")

print("✅ 라이브러리 임포트 완료")


---
# 🎬 1부. 디지털 이미지의 본질: 픽셀과 채널 (60분)

## 🧠 이론: 디지털 이미지란 무엇인가?

### 1) 픽셀(Pixel)의 본질

디지털 이미지는 결국 **숫자의 격자(grid)** 입니다.
- **픽셀(Pixel)** = Picture Element의 줄임말
- 각 픽셀은 보통 **0~255 사이의 정수**(8비트 = 2⁸ = 256단계)로 밝기를 표현
- 0 = 완전한 검정, 255 = 완전한 흰색

```
그레이스케일 이미지 (5×5)
┌──────────────────────┐
│ 10  20  30  40  50   │
│ 60  70  80  90 100   │
│110 120 130 140 150   │   ← 그냥 숫자 행렬!
│160 170 180 190 200   │
│210 220 230 240 250   │
└──────────────────────┘
```

### 2) 해상도(Resolution)

- **해상도** = 가로 픽셀 수 × 세로 픽셀 수
- 예: Full HD = 1920 × 1080 = 약 207만 픽셀
- 해상도가 높을수록 정보량 ↑, 메모리 사용량 ↑, 연산량 ↑

### 3) 비트 깊이(Bit Depth)

| 비트 | 표현 가능 단계 | 용도 |
|---|---|---|
| 1 bit | 2단계 (흑/백) | 팩스, 바코드 |
| 8 bit | 256단계 | **일반 이미지** |
| 16 bit | 65,536단계 | 의료 영상(CT, MRI) |
| 32 bit (float) | 연속값 | 딥러닝 내부 연산 |

> 💡 **딥러닝 핵심**: 학습 시에는 정수(0~255)를 **실수(float32, 0~1 또는 -1~1)**로 변환합니다. 그래야 미분이 가능하기 때문입니다.

### 4) 컬러 이미지: 채널(Channel)의 개념

컬러 이미지는 **3개의 그레이스케일 이미지가 겹쳐진 것**입니다.

- **R 채널**: 빨강 강도 (0~255)
- **G 채널**: 초록 강도 (0~255)
- **B 채널**: 파랑 강도 (0~255)

```
RGB 컬러 이미지 = R 채널 + G 채널 + B 채널
                  (H×W)     (H×W)     (H×W)
                = (H, W, 3) 형태의 3차원 배열
```

각 픽셀의 색상 = (R, G, B) 세 값의 조합으로 결정
- (255, 0, 0) → 순수 빨강
- (0, 255, 0) → 순수 초록
- (255, 255, 0) → 노랑 (R+G)
- (128, 128, 128) → 회색

## 💻 실습 1-1. 이미지 로드와 픽셀 들여다보기

먼저 인터넷에서 이미지 한 장을 불러와 픽셀 단위로 분석해보겠습니다.

In [ ]:
# 샘플 이미지 다운로드 (PIL Pillow의 공식 샘플)
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/PNG_transparency_demonstration_1.png/280px-PNG_transparency_demonstration_1.png"

# torchvision에서 제공하는 안정적인 샘플 이미지를 사용 (오프라인 대비)
from torchvision.utils import make_grid
import torchvision.datasets as datasets

# 더 안정적으로 - 직접 이미지를 만들어 보겠습니다.
# 그라디언트 패턴으로 학습용 이미지 생성
H, W = 256, 256
gradient_img = np.zeros((H, W, 3), dtype=np.uint8)
for i in range(H):
    for j in range(W):
        gradient_img[i, j, 0] = i  # R: 위에서 아래로 증가
        gradient_img[i, j, 1] = j  # G: 왼쪽에서 오른쪽으로 증가
        gradient_img[i, j, 2] = (i + j) // 2  # B: 대각선으로 증가

print(f"이미지 shape: {gradient_img.shape}")
print(f"이미지 dtype: {gradient_img.dtype}")
print(f"픽셀 값 범위: [{gradient_img.min()}, {gradient_img.max()}]")

plt.figure(figsize=(6, 6))
plt.imshow(gradient_img)
plt.title("학습용 그라디언트 이미지 (256×256×3)")
plt.axis('off')
plt.show()


In [ ]:
# 실제 이미지로도 실습해봅시다 - CIFAR-10 데이터셋의 한 장
from torchvision.datasets import CIFAR10
from torchvision import transforms

# 데이터 다운로드 (PIL Image 형태로 받기)
trainset = CIFAR10(root='./data', train=True, download=True, transform=None)

# 첫 번째 이미지 확인
sample_img, sample_label = trainset[7]  # 7번째 샘플 (말 이미지)
class_names = ['비행기', '자동차', '새', '고양이', '사슴', '개', '개구리', '말', '배', '트럭']

print(f"이미지 타입: {type(sample_img)}")
print(f"이미지 크기: {sample_img.size} (W, H)")
print(f"이미지 모드: {sample_img.mode}")
print(f"클래스: {class_names[sample_label]}")

# 시각화
plt.figure(figsize=(4, 4))
plt.imshow(sample_img)
plt.title(f"CIFAR-10 샘플: {class_names[sample_label]}")
plt.axis('off')
plt.show()


In [ ]:
# PIL Image를 NumPy 배열로 변환해 픽셀 들여다보기
img_array = np.array(sample_img)

print(f"NumPy 배열 shape: {img_array.shape}  ← (높이, 너비, 채널)")
print(f"데이터 타입: {img_array.dtype}")
print(f"전체 픽셀 수: {img_array.size:,}개")
print(f"메모리 사용량: {img_array.nbytes:,} bytes ({img_array.nbytes/1024:.2f} KB)")

print("\n📍 좌상단 5×5 영역의 R 채널 픽셀값:")
print(img_array[:5, :5, 0])

print("\n📍 정중앙 픽셀 (16, 16)의 RGB 값:")
center = img_array[16, 16]
print(f"   R={center[0]}, G={center[1]}, B={center[2]}")


## 💻 실습 1-2. RGB 채널 분리하기

컬러 이미지가 정말로 3개의 그레이스케일 이미지의 합인지 직접 확인해봅시다.

In [ ]:
# RGB 채널을 각각 분리해서 시각화
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 원본
axes[0].imshow(img_array)
axes[0].set_title(f'원본 RGB\n{img_array.shape}')
axes[0].axis('off')

# R 채널만
axes[1].imshow(img_array[:, :, 0], cmap='Reds')
axes[1].set_title(f'R 채널\n{img_array[:, :, 0].shape}')
axes[1].axis('off')

# G 채널만
axes[2].imshow(img_array[:, :, 1], cmap='Greens')
axes[2].set_title(f'G 채널\n{img_array[:, :, 1].shape}')
axes[2].axis('off')

# B 채널만
axes[3].imshow(img_array[:, :, 2], cmap='Blues')
axes[3].set_title(f'B 채널\n{img_array[:, :, 2].shape}')
axes[3].axis('off')

plt.tight_layout()
plt.show()

print("💡 각 채널은 결국 (H, W) 형태의 2D 행렬입니다.")
print("   채널이 합쳐져야 비로소 컬러 이미지가 됩니다.")


In [ ]:
# 채널별 히스토그램으로 분포 확인
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['red', 'green', 'blue']
names = ['R 채널', 'G 채널', 'B 채널']

for i, (color, name) in enumerate(zip(colors, names)):
    axes[i].hist(img_array[:, :, i].flatten(), bins=50, color=color, alpha=0.7)
    axes[i].set_title(f'{name} 픽셀값 분포')
    axes[i].set_xlabel('픽셀값 (0~255)')
    axes[i].set_ylabel('빈도')
    axes[i].set_xlim(0, 255)

plt.tight_layout()
plt.show()

print("💡 채널마다 픽셀값 분포가 다릅니다.")
print("   이것이 나중에 정규화(Normalization)에서 채널별로 평균/표준편차를 다르게 쓰는 이유입니다.")


## 💻 실습 1-3. 픽셀 직접 조작하기

이미지가 진짜 숫자 배열이라는 것을 체감하기 위해 픽셀을 직접 수정해봅시다.

In [ ]:
# 더 큰 이미지로 작업 (CIFAR-10 32×32는 너무 작음)
# 그라디언트 이미지 다시 사용
work_img = gradient_img.copy()

# 1) 빨간 사각형 그리기 (200×200~250×250 영역)
work_img[200:250, 200:250] = [255, 0, 0]  # R=255, G=0, B=0

# 2) 파란 십자가 그리기
cy, cx = 128, 128  # 중심
work_img[cy-40:cy+40, cx-5:cx+5] = [0, 0, 255]  # 세로
work_img[cy-5:cy+5, cx-40:cx+40] = [0, 0, 255]  # 가로

# 3) 우측 하단을 그레이스케일로 만들기 (상단 절반은 컬러 유지)
gray_region = work_img[150:, 50:150]  # (H_slice, W_slice, 3)
gray_value = gray_region.mean(axis=2, keepdims=True).astype(np.uint8)
work_img[150:, 50:150] = np.repeat(gray_value, 3, axis=2)

plt.figure(figsize=(8, 8))
plt.imshow(work_img)
plt.title("픽셀 직접 조작: 빨간 박스 + 파란 십자가 + 그레이 영역")
plt.axis('off')
plt.show()

print("💡 이미지 처리는 결국 NumPy 배열 슬라이싱입니다!")


## ✏️ 실습 과제 1-A

**문제:** 다음 조건을 만족하는 이미지를 직접 만들어보세요.

1. 크기: 200 × 300 (H × W)
2. 좌측 절반은 빨강(255, 0, 0), 우측 절반은 파랑(0, 0, 255)
3. 정중앙에 50×50 크기의 노란색(255, 255, 0) 사각형

> 💡 힌트: `np.zeros((H, W, 3), dtype=np.uint8)`로 시작하세요.

In [ ]:
# 여러분의 코드를 작성해보세요
# ============================================
H, W = 200, 300

# TODO: 빈 캔버스 생성
my_img = np.zeros((H, W, 3), dtype=np.uint8)

# TODO: 좌측 절반 = 빨강
# my_img[..., ...] = [255, 0, 0]

# TODO: 우측 절반 = 파랑
# my_img[..., ...] = [0, 0, 255]

# TODO: 중앙 50×50 노란색
# my_img[..., ..., :] = [255, 255, 0]

# ============================================

plt.figure(figsize=(9, 6))
plt.imshow(my_img)
plt.title("나의 첫 픽셀 아트")
plt.axis('off')
plt.show()


<details>
<summary>🔑 정답 보기 (클릭)</summary>

```python
H, W = 200, 300
my_img = np.zeros((H, W, 3), dtype=np.uint8)

# 좌측 절반 (W=0~149)
my_img[:, :W//2] = [255, 0, 0]
# 우측 절반 (W=150~299)
my_img[:, W//2:] = [0, 0, 255]
# 중앙 50×50 노랑
cy, cx = H//2, W//2
my_img[cy-25:cy+25, cx-25:cx+25] = [255, 255, 0]
```
</details>

---
# 🎬 2부. 텐서로서의 이미지 표현 (60분)

## 🧠 이론: 텐서(Tensor)가 무엇인가?

### 1) 텐서의 정의

**텐서**는 **다차원 배열**의 일반화된 이름입니다.

| 차원 | 이름 | 예시 |
|---|---|---|
| 0차원 | 스칼라(Scalar) | `3.14` |
| 1차원 | 벡터(Vector) | `[1, 2, 3]` |
| 2차원 | 행렬(Matrix) | 그레이스케일 이미지 (H×W) |
| 3차원 | 3D 텐서 | 컬러 이미지 (H×W×3) |
| 4차원 | 4D 텐서 | **이미지 배치** (N×C×H×W) |

### 2) 딥러닝 = 텐서 연산

PyTorch, TensorFlow 모두 핵심은 **텐서 연산 + 자동 미분**입니다.
- NumPy 배열과 모양이 같음
- 차이점: **GPU 가속**, **자동 미분(autograd)** 지원

### 3) 채널 순서: HWC vs CHW (★중요)

같은 이미지인데 **차원 순서가 라이브러리마다 다릅니다!**

| 표기 | 의미 | 사용처 |
|---|---|---|
| **HWC** | (Height, Width, Channels) | OpenCV, PIL, NumPy 기본 |
| **CHW** | (Channels, Height, Width) | **PyTorch 모델 입력** |

```
HWC 형태 (256, 256, 3):    CHW 형태 (3, 256, 256):
┌─────────────┐                 ┌──┬──┬──┐
│ 픽셀 1: RGB  │                 │R │G │B │
│ 픽셀 2: RGB  │                 │채│채│채│
│   ...        │                 │널│널│널│
└─────────────┘                 └──┴──┴──┘
픽셀별로 RGB 묶음              채널별로 분리됨
```

**왜 PyTorch는 CHW를 쓸까?**
- Convolution 연산 시 **채널 단위로 필터를 적용**하기 때문
- 메모리 액세스 패턴이 효율적

### 4) 배치 차원(Batch Dimension)

학습 시에는 여러 이미지를 **묶어서(batch)** 한 번에 처리합니다.

```
1장:    (3, 256, 256)        ← (C, H, W)
배치:   (32, 3, 256, 256)    ← (N, C, H, W)  N=배치 크기
```

> 💡 PyTorch 모델은 **반드시 4D 텐서 (N, C, H, W)**를 입력으로 받습니다!

### 5) 데이터 타입 변환

| 단계 | dtype | 범위 |
|---|---|---|
| 디스크/디스플레이 | `uint8` | 0~255 |
| 모델 학습 | `float32` | 0~1 또는 표준화된 값 |

## 💻 실습 2-1. NumPy 배열과 PyTorch 텐서 변환

In [ ]:
import torch

# NumPy 배열로 시작 (HWC 형태, uint8)
np_img = img_array.copy()  # (32, 32, 3)
print("=" * 50)
print("📍 NumPy 배열 (HWC, uint8)")
print("=" * 50)
print(f"shape: {np_img.shape}")
print(f"dtype: {np_img.dtype}")
print(f"range: [{np_img.min()}, {np_img.max()}]")


In [ ]:
# 1단계: NumPy → PyTorch 텐서 (단순 변환)
tensor_hwc = torch.from_numpy(np_img)
print("=" * 50)
print("📍 PyTorch 텐서 (HWC, uint8)")
print("=" * 50)
print(f"shape: {tensor_hwc.shape}")
print(f"dtype: {tensor_hwc.dtype}")
print(f"device: {tensor_hwc.device}")


In [ ]:
# 2단계: HWC → CHW 변환 (★PyTorch 모델 입력 형태)
# permute: 차원 순서 재배치 (0,1,2 → 2,0,1)
tensor_chw = tensor_hwc.permute(2, 0, 1)
print("=" * 50)
print("📍 PyTorch 텐서 (CHW, uint8)  ← 핵심!")
print("=" * 50)
print(f"shape: {tensor_chw.shape}  ← (채널, 높이, 너비)")
print(f"dtype: {tensor_chw.dtype}")


In [ ]:
# 3단계: uint8 → float32 + 정규화 (0~1)
tensor_float = tensor_chw.float() / 255.0
print("=" * 50)
print("📍 PyTorch 텐서 (CHW, float32, 0~1)")
print("=" * 50)
print(f"shape: {tensor_float.shape}")
print(f"dtype: {tensor_float.dtype}")
print(f"range: [{tensor_float.min():.4f}, {tensor_float.max():.4f}]")


In [ ]:
# 4단계: 배치 차원 추가 (★모델 입력 직전 단계)
# unsqueeze(0): 0번 위치에 차원 추가
tensor_batch = tensor_float.unsqueeze(0)
print("=" * 50)
print("📍 PyTorch 텐서 (NCHW, float32)  ← 모델 입력 가능!")
print("=" * 50)
print(f"shape: {tensor_batch.shape}  ← (배치, 채널, 높이, 너비)")
print(f"\n✅ 이제 model(tensor_batch) 형태로 모델에 넣을 수 있습니다.")


## 💻 실습 2-2. 한눈에 보는 변환 흐름

이미지 한 장이 모델에 들어가기까지의 전체 흐름을 한 번에 정리합시다.

In [ ]:
def image_to_model_input(pil_image):
    """PIL Image → PyTorch 모델 입력 (1, 3, H, W)으로 변환"""
    print("[Step 0] PIL Image")
    print(f"   → size={pil_image.size}, mode={pil_image.mode}")

    # Step 1: PIL → NumPy (HWC, uint8)
    arr = np.array(pil_image)
    print(f"\n[Step 1] NumPy (HWC, uint8)")
    print(f"   → shape={arr.shape}, dtype={arr.dtype}")

    # Step 2: NumPy → Torch (HWC, uint8)
    t = torch.from_numpy(arr)
    print(f"\n[Step 2] Torch (HWC, uint8)")
    print(f"   → shape={tuple(t.shape)}, dtype={t.dtype}")

    # Step 3: HWC → CHW
    t = t.permute(2, 0, 1)
    print(f"\n[Step 3] Torch (CHW, uint8)")
    print(f"   → shape={tuple(t.shape)}")

    # Step 4: uint8 → float32, 0~1 정규화
    t = t.float() / 255.0
    print(f"\n[Step 4] Torch (CHW, float32, 0~1)")
    print(f"   → shape={tuple(t.shape)}, dtype={t.dtype}")

    # Step 5: 배치 차원 추가
    t = t.unsqueeze(0)
    print(f"\n[Step 5] Torch (NCHW, float32)  ← 최종!")
    print(f"   → shape={tuple(t.shape)}")

    return t

model_input = image_to_model_input(sample_img)
print(f"\n🎯 최종 텐서: {model_input.shape}")


## 💻 실습 2-3. 역방향: 텐서 → 시각화

모델 출력이나 중간 결과를 시각화하려면 다시 NumPy로 되돌려야 합니다.

In [ ]:
def tensor_to_image(tensor):
    """PyTorch 텐서 (C, H, W) 또는 (1, C, H, W) → 시각화 가능한 NumPy 배열"""
    t = tensor.detach().cpu()  # GPU에 있다면 CPU로

    # 배치 차원 제거
    if t.dim() == 4:
        t = t.squeeze(0)

    # CHW → HWC
    t = t.permute(1, 2, 0)

    # 0~1 → 0~255 uint8
    arr = (t.numpy() * 255).clip(0, 255).astype(np.uint8)
    return arr

# 검증: 역변환 후 원본과 동일한지 확인
restored = tensor_to_image(model_input)
print(f"복원된 이미지 shape: {restored.shape}")
print(f"원본과 동일한가? {np.array_equal(restored, np.array(sample_img))}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(sample_img)
axes[0].set_title("원본 PIL Image")
axes[0].axis('off')
axes[1].imshow(restored)
axes[1].set_title("텐서 → 다시 NumPy로 복원")
axes[1].axis('off')
plt.show()


## 💻 실습 2-4. 미니 배치 만들기

학습은 여러 이미지를 묶어서(batch) 동시에 처리합니다.

In [ ]:
# CIFAR-10에서 8장의 이미지를 가져와 배치 만들기
from torchvision.transforms import ToTensor

to_tensor = ToTensor()  # PIL → Torch (CHW, float32, 0~1) 자동 변환

# 8장의 텐서 리스트
tensors = []
labels = []
for i in range(8):
    img, lbl = trainset[i]
    tensors.append(to_tensor(img))  # (3, 32, 32)
    labels.append(lbl)

# 배치로 합치기 (stack)
batch = torch.stack(tensors, dim=0)
print(f"배치 shape: {batch.shape}  ← (N=8, C=3, H=32, W=32)")
print(f"배치 dtype: {batch.dtype}")
print(f"라벨: {[class_names[l] for l in labels]}")

# 배치 시각화
fig, axes = plt.subplots(1, 8, figsize=(20, 3))
for i in range(8):
    img_np = tensor_to_image(batch[i])
    axes[i].imshow(img_np)
    axes[i].set_title(class_names[labels[i]])
    axes[i].axis('off')
plt.tight_layout()
plt.show()


## ✏️ 실습 과제 2-A

**문제:** 다음 변환을 한 줄씩 구현해보세요.

주어진 텐서 `x`의 shape: `(3, 224, 224)` (CHW)

1. 배치 차원 추가하여 `(1, 3, 224, 224)` 만들기
2. `(N, C, H, W)` → `(N, H, W, C)` 변환
3. float32 → uint8로 dtype 변경 (0~1 → 0~255)

In [ ]:
x = torch.rand(3, 224, 224)  # 더미 텐서
print(f"원본: {x.shape}, {x.dtype}")

# TODO 1: 배치 차원 추가
# x1 = ?
# print(f"1단계: {x1.shape}")

# TODO 2: NCHW → NHWC
# x2 = ?
# print(f"2단계: {x2.shape}")

# TODO 3: float32 → uint8
# x3 = ?
# print(f"3단계: {x3.shape}, {x3.dtype}, range=[{x3.min()}, {x3.max()}]")


<details>
<summary>🔑 정답 보기</summary>

```python
x1 = x.unsqueeze(0)              # (1, 3, 224, 224)
x2 = x1.permute(0, 2, 3, 1)      # (1, 224, 224, 3)
x3 = (x2 * 255).clamp(0, 255).to(torch.uint8)
```
</details>

---
# 🎬 3부. 이미지 전처리: 정규화와 증강 (60분)

## 🧠 이론: 왜 전처리가 필요한가?

### 1) 정규화(Normalization)가 필요한 이유

원본 이미지는 0~255 범위의 정수입니다. 이대로 학습하면 다음 문제가 생깁니다.

**문제 1: 그래디언트 폭발/소실**
- 큰 입력값 → 큰 활성화값 → 큰 그래디언트
- 학습 불안정

**문제 2: 학습 속도 저하**
- 입력 분포가 채널별로 다르면 가중치 갱신 방향이 일관되지 않음
- 등고선이 길쭉해져 최적화 경로가 비효율적

**해결: 입력을 비슷한 분포로 통일하자**
- **Min-Max Scaling**: `x / 255` → 0~1 범위
- **Standardization**: `(x - μ) / σ` → 평균 0, 표준편차 1

### 2) ImageNet 표준 정규화

대부분의 사전학습 모델은 ImageNet 통계로 학습되었습니다.

```python
mean = [0.485, 0.456, 0.406]   # R, G, B 채널별 평균
std  = [0.229, 0.224, 0.225]   # R, G, B 채널별 표준편차
```

이 값을 **항상 동일하게** 사용해야 사전학습 모델의 성능을 끌어낼 수 있습니다.

### 3) 데이터 증강(Augmentation)이 필요한 이유

**과적합(Overfitting) 방지**
- 학습 데이터가 적으면 모델이 외워버림
- 다양한 변형을 가하면 모델이 본질적인 패턴을 학습

**대표적 증강 기법**
| 기법 | 효과 |
|---|---|
| Horizontal Flip | 좌우 대칭성 학습 |
| Random Crop | 위치 불변성 |
| Color Jitter | 조명 변화 강건성 |
| Rotation | 회전 강건성 |
| Cutout / Erasing | 부분 가림 강건성 |
| Mixup / CutMix | 클래스 경계 부드럽게 |

> ⚠️ **주의**: 증강은 **학습 데이터에만** 적용. 검증/테스트는 원본 사용!

### 4) torchvision vs albumentations

| 라이브러리 | 장점 | 용도 |
|---|---|---|
| `torchvision.transforms` | PyTorch 통합, 간단 | 일반 분류 |
| `albumentations` | 빠름, 객체탐지/세그멘테이션 지원 | 고급 증강, OD/Seg |

## 💻 실습 3-1. Min-Max 정규화 직접 구현하기

In [ ]:
# 실습용 이미지: CIFAR-10에서 한 장
img_pil, lbl = trainset[10]
img_uint8 = np.array(img_pil)  # (32, 32, 3), uint8, 0~255

print(f"📍 원본 이미지")
print(f"   shape: {img_uint8.shape}")
print(f"   dtype: {img_uint8.dtype}")
print(f"   range: [{img_uint8.min()}, {img_uint8.max()}]")
print(f"   mean:  {img_uint8.mean():.2f}")
print(f"   std:   {img_uint8.std():.2f}")


In [ ]:
# Min-Max Scaling: 0~255 → 0~1
img_minmax = img_uint8.astype(np.float32) / 255.0

print(f"📍 Min-Max 정규화 (÷255)")
print(f"   range: [{img_minmax.min():.4f}, {img_minmax.max():.4f}]")
print(f"   mean:  {img_minmax.mean():.4f}")
print(f"   std:   {img_minmax.std():.4f}")


In [ ]:
# Standardization: ImageNet 통계 적용
imagenet_mean = np.array([0.485, 0.456, 0.406])
imagenet_std = np.array([0.229, 0.224, 0.225])

# 채널별로 (x - mean) / std
img_standardized = (img_minmax - imagenet_mean) / imagenet_std

print(f"📍 ImageNet 표준화")
print(f"   range: [{img_standardized.min():.4f}, {img_standardized.max():.4f}]")
print(f"   채널별 mean: {img_standardized.mean(axis=(0,1))}  ← 0에 가까움")
print(f"   채널별 std:  {img_standardized.std(axis=(0,1))}  ← 1에 가까움")


In [ ]:
# 세 가지 표현 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(img_uint8)
axes[0].set_title(f"원본 uint8\n[{img_uint8.min()}, {img_uint8.max()}]")
axes[0].axis('off')

axes[1].imshow(img_minmax)
axes[1].set_title(f"Min-Max float32\n[{img_minmax.min():.2f}, {img_minmax.max():.2f}]")
axes[1].axis('off')

# 표준화된 이미지는 음수 포함이라 시각화 시 다시 정규화
img_std_vis = (img_standardized - img_standardized.min()) / (img_standardized.max() - img_standardized.min())
axes[2].imshow(img_std_vis)
axes[2].set_title(f"ImageNet 표준화\n[{img_standardized.min():.2f}, {img_standardized.max():.2f}]")
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("💡 사람 눈에는 비슷해 보이지만, 모델 입장에서는 분포가 완전히 다릅니다!")


## 💻 실습 3-2. torchvision.transforms 활용

PyTorch 표준 전처리 파이프라인을 구축해봅니다.

In [ ]:
from torchvision import transforms

# 학습용 전처리 파이프라인 (증강 포함)
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),               # 64×64로 리사이즈
    transforms.RandomHorizontalFlip(p=0.5),    # 50% 확률 좌우 반전
    transforms.RandomRotation(degrees=15),     # ±15도 회전
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2,
        saturation=0.2, hue=0.1
    ),
    transforms.ToTensor(),                     # PIL → Tensor (CHW, 0~1)
    transforms.Normalize(                      # ImageNet 정규화
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# 검증/테스트용 (증강 없음)
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

# 동일 이미지에 학습 전처리를 9번 적용 → 매번 다른 결과
img_pil_big, _ = trainset[10]

fig, axes = plt.subplots(1, 9, figsize=(18, 2.5))
for i in range(9):
    aug_tensor = train_transform(img_pil_big)
    # 표준화 역변환 후 시각화
    aug_np = aug_tensor.numpy().transpose(1, 2, 0)
    aug_np = aug_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    aug_np = aug_np.clip(0, 1)
    axes[i].imshow(aug_np)
    axes[i].set_title(f'증강 {i+1}')
    axes[i].axis('off')
plt.suptitle("동일 이미지 → 9번 증강한 결과 (모두 다름!)", y=1.05)
plt.tight_layout()
plt.show()


## 💻 실습 3-3. albumentations 활용

산업 현장에서는 빠른 속도와 풍부한 증강 옵션 때문에 `albumentations`를 많이 씁니다.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

# albumentations 파이프라인
alb_transform = A.Compose([
    A.Resize(64, 64),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
    A.GaussNoise(p=0.3),  # 가우시안 노이즈 (현장 환경 모사)
    A.CoarseDropout(num_holes_range=(1,3), hole_height_range=(8,16),
                    hole_width_range=(8,16), p=0.5),  # 부분 가림 (Cutout)
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# albumentations은 NumPy 배열을 입력으로 받음 (HWC, uint8)
img_np_big = np.array(img_pil_big)

fig, axes = plt.subplots(1, 9, figsize=(18, 2.5))
for i in range(9):
    result = alb_transform(image=img_np_big)
    aug_tensor = result['image']  # CHW
    aug_np = aug_tensor.numpy().transpose(1, 2, 0)
    aug_np = aug_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
    aug_np = aug_np.clip(0, 1)
    axes[i].imshow(aug_np)
    axes[i].set_title(f'Alb {i+1}')
    axes[i].axis('off')
plt.suptitle("albumentations: 더 강력한 증강 (CoarseDropout, GaussNoise 포함)", y=1.05)
plt.tight_layout()
plt.show()


## 💻 실습 3-4. 증강의 효과 비교 - 속도 측정

In [ ]:
import time

# 동일 이미지에 100번 증강 적용 - 속도 비교
N = 100

# torchvision
start = time.time()
for _ in range(N):
    _ = train_transform(img_pil_big)
tv_time = time.time() - start

# albumentations
start = time.time()
for _ in range(N):
    _ = alb_transform(image=img_np_big)
alb_time = time.time() - start

print(f"📊 100회 증강 속도 비교")
print(f"   torchvision    : {tv_time*1000:.1f} ms ({tv_time*10:.2f} ms/장)")
print(f"   albumentations : {alb_time*1000:.1f} ms ({alb_time*10:.2f} ms/장)")
print(f"   속도 차이: albumentations가 {tv_time/alb_time:.2f}배 빠름")
print()
print("💡 대용량 데이터셋(수백만 장) 학습 시에는 albumentations가 유리합니다.")


## ✏️ 실습 과제 3-A

**문제:** 다음 요구사항에 맞는 `torchvision.transforms.Compose` 파이프라인을 작성하세요.

1. 이미지를 224×224로 리사이즈
2. 30% 확률로 좌우 반전
3. ±10도 회전
4. ToTensor
5. ImageNet 정규화

In [ ]:
# TODO: 파이프라인 작성
# my_transform = transforms.Compose([
#     ...
# ])

# 테스트
# result = my_transform(img_pil_big)
# print(f"결과 shape: {result.shape}")
# print(f"결과 dtype: {result.dtype}")


---
# 🎬 4부. MLP vs CNN: Spatial Locality (60분)

## 🧠 이론: 왜 이미지에는 CNN인가?

### 1) MLP(Multi-Layer Perceptron)의 한계

MLP는 입력을 **1차원 벡터로 펼쳐서(flatten)** 처리합니다.

```
이미지 (28×28)         MLP 입력
┌─────────┐
│ █ █ █ . │           [0, 1, 1, 0, 1, 0, 0, ..., 1]
│ . █ . . │   →
│ █ █ . . │           ← 784차원 벡터로 일렬 펼치기
│ . . . . │
└─────────┘
```

**문제점 3가지:**

❌ **공간 정보 파괴**
- 인접한 픽셀이 분리됨 (예: (0,0)과 (0,1) 픽셀의 관계 손실)
- "이웃 픽셀끼리는 관련 있다"는 사실을 모델이 처음부터 다시 학습해야 함

❌ **파라미터 폭발**
- 28×28 이미지 → 첫 은닉층(512 뉴런)만 해도 가중치 = 784 × 512 = **40만 개**
- 224×224 이미지면? 224×224 × 512 = **2,567만 개** (한 층에!)

❌ **위치 변화에 취약 (No Translation Invariance)**
- 같은 고양이가 좌상단에 있는 것과 우하단에 있는 것을 완전히 다른 패턴으로 인식

### 2) CNN의 핵심 3대 원리

CNN은 이미지의 **본질적 특성**을 모델 구조에 반영합니다.

#### (1) **Spatial Locality (공간 지역성)**
> "픽셀은 가까운 이웃과 관련이 깊다"

작은 필터(예: 3×3)로 **국소 영역만** 보고 특징을 뽑음. 멀리 떨어진 픽셀은 처음부터 무시.

#### (2) **Parameter Sharing (가중치 공유)**
> "같은 필터를 이미지 전체에 슬라이딩하며 적용"

3×3 필터 한 개 = **9개 파라미터**로 이미지 어디서든 동일한 특징 검출.
→ 좌상단의 모서리 검출기 = 우하단의 모서리 검출기

#### (3) **Translation Invariance (위치 불변성)**
> "고양이가 어디 있든 고양이"

가중치 공유 덕분에 객체가 어디로 이동해도 동일한 응답.

### 3) 파라미터 수 비교

**과제: 28×28 → 32 출력 채널 변환**

| 방식 | 가중치 수 |
|---|---|
| Flatten + Dense(32) | 28 × 28 × 32 = **25,088** |
| Conv2d(1, 32, 3×3)  | 1 × 32 × 3 × 3 = **288** |

**87배 차이!** 그러면서도 CNN이 더 잘 작동합니다.

### 4) 실험으로 검증해봅시다

이제 MLP와 CNN을 직접 만들어 같은 데이터로 학습시키고 비교해보겠습니다.

## 💻 실습 4-1. MNIST 데이터 준비

In [ ]:
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader

# MNIST: 28×28 흑백 손글씨 숫자 (10클래스)
mnist_transform = transforms.Compose([
    transforms.ToTensor(),                # (1, 28, 28), 0~1
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST 통계
])

mnist_train = MNIST('./data', train=True, download=True, transform=mnist_transform)
mnist_test = MNIST('./data', train=False, download=True, transform=mnist_transform)

print(f"학습 데이터: {len(mnist_train)}장")
print(f"테스트 데이터: {len(mnist_test)}장")
print(f"이미지 shape: {mnist_train[0][0].shape}")

# 샘플 시각화
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img, lbl = mnist_train[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"라벨: {lbl}")
    ax.axis('off')
plt.tight_layout()
plt.show()


In [ ]:
BATCH_SIZE = 128
train_loader = DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(mnist_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 한 배치 확인
imgs, lbls = next(iter(train_loader))
print(f"배치 shape: {imgs.shape}  ← (N, C, H, W)")
print(f"라벨 shape: {lbls.shape}")


## 💻 실습 4-2. MLP 모델 정의

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleMLP(nn.Module):
    """
    이미지를 일렬로 펼쳐서 처리하는 MLP
    구조: 784 → 512 → 256 → 10
    """
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()           # (N, 1, 28, 28) → (N, 784)
        self.fc1 = nn.Linear(28*28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

mlp = SimpleMLP().to(device)
print(mlp)

# 파라미터 수 계산
mlp_params = sum(p.numel() for p in mlp.parameters())
print(f"\n📊 MLP 파라미터 수: {mlp_params:,} 개")


## 💻 실습 4-3. CNN 모델 정의

In [ ]:
class SimpleCNN(nn.Module):
    """
    공간 구조를 유지하며 처리하는 CNN
    구조: Conv(1→32) → Pool → Conv(32→64) → Pool → FC(10)
    """
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)   # (N, 32, 28, 28)
        self.pool1 = nn.MaxPool2d(2)                              # (N, 32, 14, 14)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)  # (N, 64, 14, 14)
        self.pool2 = nn.MaxPool2d(2)                              # (N, 64, 7, 7)
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(64*7*7, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool1(x)
        x = F.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.fc(x)
        return x

cnn = SimpleCNN().to(device)
print(cnn)

cnn_params = sum(p.numel() for p in cnn.parameters())
print(f"\n📊 CNN 파라미터 수: {cnn_params:,} 개")
print(f"\n💥 비교: MLP({mlp_params:,}) vs CNN({cnn_params:,})")
print(f"   CNN이 MLP보다 {mlp_params/cnn_params:.2f}배 적은 파라미터!")


## 💻 실습 4-4. 학습 함수 정의 및 두 모델 학습

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    return total_loss/total, correct/total

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        out = model(imgs)
        loss = criterion(out, lbls)
        total_loss += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == lbls).sum().item()
        total += imgs.size(0)
    return total_loss/total, correct/total

def train_model(model, name, epochs=3):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss':[], 'train_acc':[], 'test_loss':[], 'test_acc':[]}

    print(f"\n{'='*50}")
    print(f"🚀 {name} 학습 시작 (총 {epochs} epochs)")
    print(f"{'='*50}")

    for epoch in range(1, epochs+1):
        tl, ta = train_one_epoch(model, train_loader, optimizer, criterion)
        vl, va = evaluate(model, test_loader, criterion)
        history['train_loss'].append(tl); history['train_acc'].append(ta)
        history['test_loss'].append(vl); history['test_acc'].append(va)
        print(f"Epoch {epoch}: train_loss={tl:.4f}, train_acc={ta:.4f} | "
              f"test_loss={vl:.4f}, test_acc={va:.4f}")
    return history

mlp_hist = train_model(mlp, "MLP", epochs=3)
cnn_hist = train_model(cnn, "CNN", epochs=3)


## 💻 실습 4-5. 학습 곡선 비교

In [ ]:
epochs = range(1, 4)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(epochs, mlp_hist['test_loss'], 'o-', label='MLP test loss')
axes[0].plot(epochs, cnn_hist['test_loss'], 's-', label='CNN test loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('테스트 손실 비교')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(epochs, mlp_hist['test_acc'], 'o-', label='MLP test acc')
axes[1].plot(epochs, cnn_hist['test_acc'], 's-', label='CNN test acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('테스트 정확도 비교')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🏆 최종 비교")
print(f"   MLP: 파라미터 {mlp_params:,} 개, 최종 정확도 {mlp_hist['test_acc'][-1]:.4f}")
print(f"   CNN: 파라미터 {cnn_params:,} 개, 최종 정확도 {cnn_hist['test_acc'][-1]:.4f}")
print(f"\n💡 CNN이 적은 파라미터로 더 높은 성능을 보입니다.")
print(f"   이것이 바로 이미지에 적합한 inductive bias의 위력입니다!")


## 💻 실습 4-6. Translation Invariance 직접 검증

이번에는 **이미지를 옆으로 옮겼을 때** 두 모델이 어떻게 반응하는지 봅시다.

In [ ]:
# 테스트 이미지 한 장 선택
test_img, test_lbl = mnist_test[0]
print(f"원본 라벨: {test_lbl}")

# 다양한 위치로 평행 이동시킨 이미지 만들기
def shift_image(img_tensor, dx, dy):
    """img_tensor: (1, 28, 28) → 평행이동된 이미지"""
    img = img_tensor.squeeze().numpy()
    shifted = np.zeros_like(img)
    H, W = img.shape

    # 이동 후 유효 영역 계산
    src_y_start = max(0, -dy)
    src_y_end = min(H, H - dy)
    src_x_start = max(0, -dx)
    src_x_end = min(W, W - dx)
    dst_y_start = max(0, dy)
    dst_y_end = min(H, H + dy)
    dst_x_start = max(0, dx)
    dst_x_end = min(W, W + dx)

    shifted[dst_y_start:dst_y_end, dst_x_start:dst_x_end] = \
        img[src_y_start:src_y_end, src_x_start:src_x_end]

    # MNIST 정규화 적용된 상태로 반환
    return torch.from_numpy(shifted).unsqueeze(0).float()

# 정규화 해제 후 다시 정규화 (시각화용)
mean, std = 0.1307, 0.3081
denorm_img = test_img * std + mean

# 다양한 이동 거리로 테스트
shifts = [-7, -4, -2, 0, 2, 4, 7]
shifted_imgs = [shift_image(denorm_img, dx, 0) for dx in shifts]

fig, axes = plt.subplots(1, len(shifts), figsize=(14, 2.5))
for ax, img, dx in zip(axes, shifted_imgs, shifts):
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"dx={dx}")
    ax.axis('off')
plt.suptitle("동일한 숫자를 좌우로 이동시킨 이미지들", y=1.05)
plt.tight_layout()
plt.show()


In [ ]:
# 각 모델이 이동된 이미지를 얼마나 잘 인식하는지 측정
mlp.eval(); cnn.eval()

mlp_confidences = []
cnn_confidences = []

for dx in shifts:
    shifted = shift_image(denorm_img, dx, 0)
    # 다시 정규화
    shifted_norm = (shifted - mean) / std
    inp = shifted_norm.unsqueeze(0).to(device)  # (1, 1, 28, 28)

    with torch.no_grad():
        mlp_out = F.softmax(mlp(inp), dim=1)[0, test_lbl].item()
        cnn_out = F.softmax(cnn(inp), dim=1)[0, test_lbl].item()

    mlp_confidences.append(mlp_out)
    cnn_confidences.append(cnn_out)

# 시각화
plt.figure(figsize=(10, 5))
plt.plot(shifts, mlp_confidences, 'o-', linewidth=2, markersize=10, label='MLP')
plt.plot(shifts, cnn_confidences, 's-', linewidth=2, markersize=10, label='CNN')
plt.xlabel('이동 거리 (픽셀)')
plt.ylabel(f'정답 클래스({test_lbl}) 확률')
plt.title('Translation Invariance 검증: 이미지를 옆으로 이동시킬수록...')
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.ylim(-0.05, 1.05)
plt.show()

print("💡 관찰 포인트:")
print("   - MLP는 이동에 따라 확률이 급격히 떨어집니다.")
print("   - CNN은 상대적으로 안정적인 확률을 유지합니다.")
print("   - 이것이 바로 Translation Invariance의 효과입니다.")


## 💻 실습 4-7. CNN 필터(가중치) 시각화 - 보너스

CNN이 학습한 필터를 직접 들여다봅시다. 학습된 필터가 어떤 시각 패턴을 검출하는지 알 수 있습니다.

In [ ]:
# 첫 번째 Conv 레이어의 가중치 추출
conv1_weights = cnn.conv1.weight.data.cpu()  # (32, 1, 3, 3)
print(f"Conv1 필터 shape: {conv1_weights.shape}")
print(f"   → 32개의 3×3 필터 (입력 채널 1)")

# 32개 필터 시각화
fig, axes = plt.subplots(4, 8, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    f = conv1_weights[i, 0].numpy()
    ax.imshow(f, cmap='RdBu_r', vmin=-f.max(), vmax=f.max())
    ax.set_title(f'#{i}', fontsize=8)
    ax.axis('off')
plt.suptitle('CNN이 학습한 32개의 3×3 필터 (Conv1 레이어)', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 빨간색은 양의 가중치, 파란색은 음의 가중치입니다.")
print("   필터마다 서로 다른 모서리·곡선·패턴 검출기 역할을 학습했습니다.")


---
# 🎯 미니 프로젝트: 종합 실습

## 과제: CIFAR-10에서 MLP vs CNN 비교

지금까지 배운 내용을 종합하여, CIFAR-10(컬러 이미지)에 대해 MLP와 CNN의 성능을 비교해보세요.

### 요구사항

1. **데이터 전처리**
   - Resize 32×32, ToTensor, ImageNet 정규화 적용
   - 학습 데이터에는 RandomHorizontalFlip 추가

2. **MLP 모델**
   - 구조: 3072(=3×32×32) → 1024 → 512 → 10
   - 파라미터 수 출력

3. **CNN 모델**
   - 구조: Conv(3→32) → Pool → Conv(32→64) → Pool → Conv(64→128) → Pool → FC(10)
   - 파라미터 수 출력

4. **학습 및 비교**
   - 5 epochs 학습
   - 두 모델의 학습 곡선 비교 시각화
   - 결론 작성: "왜 CNN이 더 좋은가?"


In [ ]:
# 미니 프로젝트 - 여러분의 코드를 작성해보세요

# Step 1: 전처리 파이프라인
# cifar_train_transform = ...
# cifar_test_transform = ...

# Step 2: 데이터 로더
# cifar_train = CIFAR10('./data', train=True, ...)
# ...

# Step 3: MLP 모델
# class CIFAR_MLP(nn.Module): ...

# Step 4: CNN 모델
# class CIFAR_CNN(nn.Module): ...

# Step 5: 학습 + 비교 시각화

# (작성 후 강사와 함께 리뷰)


---
# 📝 1주차 1-1 마무리

## ✅ 핵심 정리

### 1. 디지털 이미지의 본질
- 이미지 = **숫자의 격자(grid)**, 픽셀 단위로 0~255 정수값
- 컬러 이미지 = R/G/B 3개 채널이 겹쳐진 것
- Shape: `(H, W)` 그레이스케일, `(H, W, 3)` 컬러

### 2. 텐서 표현
- NumPy/PIL: **HWC** 순서 (H, W, C)
- PyTorch: **CHW** 순서 (C, H, W), 학습 시 **NCHW** (N, C, H, W)
- 핵심 변환: `tensor.permute(2, 0, 1)`, `tensor.unsqueeze(0)`

### 3. 전처리 파이프라인
- **정규화**: 0~255 → 0~1 → ImageNet 표준화 (mean=[0.485, 0.456, 0.406])
- **증강**: Flip, Rotation, ColorJitter, Cutout 등 (★학습 데이터에만)
- 표준 라이브러리: `torchvision.transforms`, `albumentations`

### 4. MLP vs CNN
- **MLP의 한계**: 공간 정보 파괴, 파라미터 폭발, 위치 변화에 취약
- **CNN의 3대 원리**: Spatial Locality, Parameter Sharing, Translation Invariance
- **결과**: CNN이 적은 파라미터로 더 높은 성능 달성

## 📚 다음 차시 예고

**1-2. Convolution & Pooling 원리** (4시간)
- Convolution 연산의 stride, padding, dilation
- Max/Avg Pooling의 의미
- ReLU, GELU, Swish 활성화 함수
- NumPy로 Convolution 직접 구현

## 🔗 추천 자료

- 📖 [PyTorch 공식 튜토리얼 - 텐서](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
- 📖 [CS231n - Image Classification](https://cs231n.github.io/classification/)
- 📖 [albumentations 공식 문서](https://albumentations.ai/docs/)
- 🎥 [3Blue1Brown - Neural Networks](https://www.youtube.com/watch?v=aircAruvnKk)

---

**수고하셨습니다! 🎉**

질문이 있다면 강사에게 자유롭게 문의하세요.
